# 21. 공동 DB 적재 (예방·법령 Graph, 2026-11-05 회의 결정 반영)

- 승인 근거: 회의 결정 A·B(새 ID 2개), C(최종 결과 정리 → 충돌 확인 → 적재), E(basis EXPLICIT), 사용자 승인 "응".
- 범위: Document 4, Chunk 31(임베딩 포함), SchoolRole 1, ActionConcept 1, SchoolAction 3, 관계 43.
- 안전: 한 트랜잭션, CREATE 전용, 쌍 단위 검사. 실패하면 전부 되돌린다.
- 재실행 확인: 같은 데이터를 다시 넣어 제약 오류가 나고 개수가 늘지 않는지 본다.

In [1]:
import sys
sys.path.insert(0, "../../src")
from rag.prevention_legal.load_graph import build_rows, check_input, embed_chunks, load, TARGET
data = build_rows()
check_input(data)
print("입력 확인 통과:", {k: len(v) for k, v in data.items()})
vectors = embed_chunks(data["chunks"])
print("임베딩:", len(vectors), "개 /", len(vectors[0]), "차원")

입력 확인 통과: {'docs': 4, 'chunks': 31, 'actions': 3}


임베딩: 31 개 / 1536 차원


In [2]:
result = load(data, vectors)
print("적재 결과 (검사 통과):", result)

적재 결과 (검사 통과): {'nodes': {'Document': 4, 'Chunk': 31, 'SchoolRole': 1, 'ActionConcept': 1, 'SchoolAction': 3}, 'relations': {'HAS_CHUNK': 31, 'SUPPORTED_BY': 3, 'FOR_DISEASE': 3, 'DONE_BY': 3, 'INSTANCE_OF': 3}}


In [3]:
from rag.prevention_legal.neo4j_conn import connect
d = connect()
with d.session() as s:
    print("Document 우리 문서:", s.run("MATCH (n:Document) WHERE n.document_id IN $ids RETURN count(n) AS n",
                                       ids=[x["document_id"] for x in data["docs"]]).single()["n"])
    print("Chunk 우리 문서:", s.run("MATCH (n:Chunk) WHERE n.chunk_id IN $ids RETURN count(n) AS n",
                                    ids=[x["chunk_id"] for x in data["chunks"]]).single()["n"])
    print("Chunk 중 벡터 없음:", s.run("MATCH (n:Chunk) WHERE n.chunk_id IN $ids AND n.embedding IS NULL RETURN count(n) AS n",
                                       ids=[x["chunk_id"] for x in data["chunks"]]).single()["n"])
    print("SchoolAction 우리 행동:", s.run("MATCH (n:SchoolAction) WHERE n.action_id IN $ids RETURN count(n) AS n",
                                          ids=[x["action_id"] for x in data["actions"]]).single()["n"])
    print("전체 Chunk:", s.run("MATCH (n:Chunk) RETURN count(n) AS n").single()["n"])
d.close()

Document 우리 문서: 4
Chunk 우리 문서: 31


Chunk 중 벡터 없음: 0
SchoolAction 우리 행동: 3


전체 Chunk: 1567


In [4]:
# 재실행 확인: 같은 데이터를 다시 넣으면 제약 오류가 나야 하고, 개수는 그대로여야 한다
from neo4j.exceptions import ConstraintError
try:
    load(data, vectors)
    print("재실행이 성공함 (문제)")
except ConstraintError as e:
    print("재실행 실패 (기대):", type(e).__name__)
d = connect()
with d.session() as s:
    print("전체 Chunk (재실행 후):", s.run("MATCH (n:Chunk) RETURN count(n) AS n").single()["n"])
    print("우리 Chunk (재실행 후):", s.run("MATCH (n:Chunk) WHERE n.chunk_id IN $ids RETURN count(n) AS n",
                                          ids=[x["chunk_id"] for x in data["chunks"]]).single()["n"])
d.close()

재실행 실패 (기대): ConstraintError


전체 Chunk (재실행 후): 1567
우리 Chunk (재실행 후): 31
